# Capítulo 3 — Amostragem e Estimação

Notebook com o **código** deste capítulo, pronto para o Google Colab. A explicação de cada trecho está no site do livro; aqui você roda os exemplos.

Rode a célula de **setup** abaixo primeiro (uma vez), depois as demais em ordem.

In [ ]:
# Setup (rode uma vez).
!curl -sO https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/formato.py   # baixa o ajudante de formatação do livro

## 3.1 — Amostragem Aleatória e Viés de Amostra

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
renda = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/loans_income.csv").rename(columns={"x": "Renda"})["Renda"]

In [ ]:
print(f"População: {num(len(renda), 0)} rendas")
print(f"Média:   {num(renda.mean(), 0)}")
print(f"Mediana: {num(renda.median(), 0)}")

In [ ]:
amostra = renda.sample(100, random_state=42)
print(f"Média da amostra (n=100): {num(amostra.mean(), 0)}")
print(f"Média da população:       {num(renda.mean(), 0)}")

## 3.2 — Viés de Seleção

In [ ]:
import numpy as np
from formato import num

In [ ]:
rng = np.random.default_rng(42)

# 100 pessoas jogam uma moeda JUSTA 20 vezes cada. Quantas caras a "melhor" tira?
tentativas = [int(rng.integers(0, 2, 20).sum()) for _ in range(100)]
melhor = max(tentativas)
print(f"Em 100 tentativas de 20 lançamentos de uma moeda justa,")
print(f"a melhor sequência deu {melhor} caras de 20.")

## 3.3 — Distribuição Amostral de uma Estatística

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
renda = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/loans_income.csv").rename(columns={"x": "Renda"})["Renda"]

In [ ]:
rng_medias = lambda n: [renda.sample(n, random_state=i).mean() for i in range(1000)]

fig, eixos = plt.subplots(1, 3, figsize=(11, 3.3), sharex=True)
for ax, (n, dados, titulo) in zip(eixos, [
    (1, renda.sample(1000, random_state=0), "Dados (n=1)"),
    (5, rng_medias(5), "Médias de n=5"),
    (20, rng_medias(20), "Médias de n=20"),
]):
    ax.hist(dados, bins=40, color="#b0c4d8", edgecolor="white")
    ax.set_title(titulo, fontsize=10)
    ax.set_xlabel("Renda (US$)")
eixos[0].set_ylabel("Frequência")
plt.tight_layout()
plt.show()

In [ ]:
dp_pop = renda.std(ddof=1)
print(f"{'n':>4}  {'desvio das médias':>18}  {'σ/√n (teórico)':>16}")
for n in [1, 5, 20, 100]:
    medias = [renda.sample(n, random_state=i).mean() for i in range(1000)]
    print(f"{n:>4}  {num(np.std(medias, ddof=1), 0):>18}  {num(dp_pop / np.sqrt(n), 0):>16}")

## 3.4 — Bootstrap

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
renda = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/loans_income.csv").rename(columns={"x": "Renda"})["Renda"]

In [ ]:
amostra = renda.sample(1000, random_state=1)

n_boot = 1000
medianas = [amostra.sample(len(amostra), replace=True, random_state=i).median()
            for i in range(n_boot)]

print(f"Mediana da amostra:      {num(amostra.median(), 0)}")
print(f"Erro padrão (bootstrap): {num(np.std(medianas, ddof=1), 2)}")

In [ ]:
fig, ax = plt.subplots()
ax.hist(medianas, bins=30, color="#b0c4d8", edgecolor="white")
ax.axvline(amostra.median(), color="#c0392b", linewidth=2, label=f"Mediana da amostra: {num(amostra.median(), 0)}")
ax.set_xlabel("Mediana reamostrada (US$)")
ax.set_ylabel("Frequência")
ax.legend()
plt.tight_layout()
plt.show()

## 3.5 — Intervalos de Confiança

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from formato import num

plt.rcParams["figure.figsize"] = (7, 4)
renda = pd.read_csv("https://raw.githubusercontent.com/BragaD/UnDF-Bases3-Estatistica-202602/main/dados/loans_income.csv").rename(columns={"x": "Renda"})["Renda"]

In [ ]:
amostra20 = renda.sample(20, random_state=3)
print(f"Média da amostra (n=20): {num(amostra20.mean(), 0)}")

boot = [amostra20.sample(20, replace=True, random_state=i).mean() for i in range(1000)]
lo, hi = np.percentile(boot, [5, 95])
print(f"IC de 90%: [{num(lo, 0)}, {num(hi, 0)}]")
print(f"Largura: {num(hi - lo, 0)}")

In [ ]:
fig, ax = plt.subplots()
ax.hist(boot, bins=30, color="#b0c4d8", edgecolor="white")
ax.axvline(lo, color="#c0392b", linewidth=2, linestyle="--", label=f"IC 90%: [{num(lo, 0)}, {num(hi, 0)}]")
ax.axvline(hi, color="#c0392b", linewidth=2, linestyle="--")
ax.axvline(amostra20.mean(), color="#27ae60", linewidth=2, label=f"Média da amostra: {num(amostra20.mean(), 0)}")
ax.set_xlabel("Média reamostrada (US$)")
ax.set_ylabel("Frequência")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
lo95, hi95 = np.percentile(boot, [2.5, 97.5])
print(f"IC de 90%: largura {num(hi - lo, 0)}")
print(f"IC de 95%: largura {num(hi95 - lo95, 0)}")